<a href="https://colab.research.google.com/github/zFonta/CEIA-TF-Chess-DL/blob/main/notebooks/08_motor_y_partidas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08 - El motor y sus partidas

Cierra la tarea **4.7** de lo que quedaba abierto, el **bloque 5** (motor) y el
**bloque 6** (evaluación del sistema).

Hasta acá el trabajo midió cuánto se parecen las redes a Stockfish. Esta
notebook mide algo distinto: **cuánto juegan**. No son la misma pregunta, y esa
diferencia es el contenido del bloque 6.

## Qué se responde acá

| | Qué |
|---|---|
| **4.7** | Tiempo de inferencia por lote y desglose de métricas por control de tiempo |
| **1.6** | Selección de jugada por búsqueda de un nivel sobre la evaluación |
| **1.7** | Tiempo por jugada, medido, contra el presupuesto de 5 segundos |
| **4.2** | Evaluación reportada en centipeones y en vista de las blancas |
| **Bloque 6** | Elo estimado, pérdida media en centipeones y acuerdo de jugada |

## Lo que hay que tener presente al leer los resultados

**La búsqueda de un nivel es la del plan; 2 y 3 son extensión.** Cada ply
multiplica las posiciones a evaluar por la ramificación —unas 33—, así que el
costo crece rápido y sólo se corren los torneos a las profundidades que entren
en los 5 segundos.

**Profundidad 2 arregla algo concreto.** A un ply el árbol termina en la jugada
propia, así que la recaptura no está en el árbol: el motor gana una torre con la
dama y no ve que el rey se la come. A dos plies termina después de la respuesta
del rival, que es exactamente ese punto ciego. Las profundidades impares vuelven
a terminar en jugada propia y son estructuralmente optimistas.

**Cien partidas no alcanzan para separar dos motores parecidos.** La tasa de
puntos de cien partidas trae varios puntos porcentuales de incertidumbre. Por eso
la métrica principal del bloque es la **pérdida media en centipeones**: preguntarle
a Stockfish, sobre miles de posiciones, cuánto tira cada jugada elegida contra la
mejor. Es continua, cada posición cuenta, y el intervalo es mucho más angosto.

> **Runtime de GPU.** Stockfish es CPU, pero el tiempo lo domina la inferencia de
> la red: en las mediciones previas la red se llevó el 96 % y el recorrido del
> árbol el 4 %.

## 1. Entorno

In [1]:
# Clonar el repositorio e instalar el paquete.
# Idempotente: se puede volver a correr tal cual despues de una desconexion.
import os, sys, subprocess, importlib
from pathlib import Path

REPO_DIR = Path("/content/CEIA-TF-Chess-DL")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/zFonta/CEIA-TF-Chess-DL.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# pip registra el paquete editable con un .pth, y los .pth solo se procesan al
# arrancar el interprete: un kernel que ya esta corriendo no lo ve. Agregar src/
# a sys.path lo hace visible sin tener que reiniciar el runtime.
src = str(REPO_DIR / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()

# Stockfish con version fija. Entrenar no lo usa, pero sin el se saltean los 17
# tests de integracion del pipeline, que son la evidencia del requerimiento 3.2.
subprocess.run(["bash", "scripts/setup_stockfish.sh"], check=True)
os.environ["PATH"] = f"{REPO_DIR}/bin:" + os.environ["PATH"]

print("Listo. Directorio de trabajo:", os.getcwd())

Listo. Directorio de trabajo: /content/CEIA-TF-Chess-DL


In [2]:
import numpy as np
import torch
from chessdl.colab import TRAINING, describe_runtime

runtime = describe_runtime(phase=TRAINING)
print("GPU  :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")
for aviso in runtime.warnings():
    print("AVISO:", aviso)

GPU  : Tesla T4


## 2. Tests

In [3]:
!{sys.executable} -m pytest -q

........................................................................ [ 14%]
........................................................................ [ 29%]
........................................................................ [ 44%]
........................................s............................... [ 59%]
........................................................................ [ 74%]
........................................................................ [ 88%]
......................................................                   [100%]
485 passed, 1 skipped in 80.80s (0:01:20)


## 3. Dataset y partición

Se necesita el split de **test** dos veces: para el desglose de la 4.7, y como
fuente de las posiciones de apertura de las partidas. Son posiciones reales que
ningún modelo vio durante el entrenamiento.

In [4]:
from chessdl import hf
from chessdl.config import load_config
from chessdl.data import schema
from chessdl.training.cache import build_cache, cache_path_for, load_cache
from chessdl.training.split import leaked_games, split_masks

cfg = load_config()
token = hf.get_token()
directorio = hf.download_dataset(cfg.output.hf_repo_id, "/content/ceia-chess/hub")
tabla = schema.read_dataset(schema.shard_paths(directorio))

fens     = tabla["fen"].to_pylist()
game_ids = tabla["game_id"].to_pylist()
plies    = np.asarray(tabla["ply"])
controles = np.asarray(tabla["time_control"])
targets  = np.asarray(tabla["value_stm"], dtype=np.float32)

masks = split_masks(game_ids, cfg.training.split_config())
assert not leaked_games(masks, game_ids)
idx_test = np.flatnonzero(masks['test'])

ruta_cache = cache_path_for(cfg.training.cache_dir)
if not ruta_cache.exists():
    build_cache(fens, ruta_cache, progress=True)
cache = load_cache(ruta_cache, expected_rows=len(fens))

print(f"test: {len(idx_test):,} posiciones")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 158 files:   0%|          | 0/158 [00:00<?, ?it/s]

test: 127,499 posiciones


## 4. Los dos modelos entrenados

Se bajan del Hub los mejores checkpoints por validación de cada arquitectura. El
`model_config` viaja dentro del checkpoint, así que la red se reconstruye con la
forma exacta con la que fue entrenada — cargar pesos en una arquitectura
compatible pero distinta no siempre falla, y cuando no falla es peor.

In [5]:
from chessdl.engine.evaluator import Evaluator
from chessdl.engine.loader import load_from_hub

repo_modelos = f'{cfg.output.hf_namespace}/{cfg.training.hf_models_repo}'
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'

CORRIDAS = {
    'ResNet':      'campana2-warmup',
    'Transformer': 'transformer-campana2-lotes-chicos',
}

evaluadores = {}
for nombre, corrida in CORRIDAS.items():
    modelo = load_from_hub(repo_modelos, corrida, token=token, device=dispositivo)
    evaluadores[nombre] = Evaluator(modelo, device=dispositivo)
    print(f'{nombre:<12}{evaluadores[nombre].describe()}')

ResNet      ChessResNet en cuda (2,913,345 parametros, lotes de 512)
Transformer ChessTransformer en cuda (2,735,361 parametros, lotes de 512)


## 5. Lo que faltaba de la tarea 4.7

### 5.1 Tiempo de inferencia por lote

El dato que el documento de alcance dejó anotado como "gratis de medir, y le
ahorra trabajo al bloque 5".

In [6]:
import time
from chessdl.training.cache import cached_to_tensor

muestra = torch.from_numpy(cached_to_tensor(np.asarray(cache[idx_test[:4096]]))).to(dispositivo)

print(f"{'arquitectura':<14}{'lote':>7}{'ms':>10}{'pos/s':>12}")
print('-' * 43)
rendimiento = {}
for nombre, ev in evaluadores.items():
    for n in (32, 256, 2048):
        lote = muestra[:n]
        with torch.no_grad():
            for _ in range(3):
                ev.model(lote)                      # calentamiento
            if dispositivo == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(10):
                ev.model(lote)
            if dispositivo == 'cuda':
                torch.cuda.synchronize()
            dt = (time.perf_counter() - t0) / 10
        print(f"{nombre if n == 32 else '':<14}{n:>7}{dt*1000:>10.1f}{n/dt:>12,.0f}")
        rendimiento[(nombre, n)] = n / dt

arquitectura     lote        ms       pos/s
-------------------------------------------
ResNet             32       2.9      10,992
                  256      16.6      15,446
                 2048     105.1      19,482
Transformer        32       5.4       5,952
                  256      43.6       5,871
                 2048     365.0       5,611


### 5.2 Desglose por control de tiempo

El dataset es **91,4 % Blitz** y Classical aporta el **0,29 %**. La pregunta que
el dataset card dejó anotada es si el modelo rinde peor en los ritmos que casi no
vio.

> **Cuidado con la última fila.** Classical son ~370 posiciones de test. Alcanza
> para un número, no para una conclusión fina: sólo una diferencia grande sería
> distinguible del ruido con esa muestra, y la columna de tamaño está al lado
> justamente para que se lea así.

In [7]:
from chessdl.training.loop import predict
from chessdl.training.metrics import evaluate

orden = np.sort(idx_test)
predicciones = {n: predict(ev.model, cache, targets, orden, dispositivo)
                for n, ev in evaluadores.items()}

print(f"{'control':<14}{'posiciones':>12}{'RMSE':>10}{'MAE cp':>9}{'signo':>9}")
print('-' * 54)
for nombre, pred in predicciones.items():
    print(nombre)
    for control in ['Blitz', 'Rapid', 'Classical']:
        sel = controles[orden] == control
        if sel.sum() == 0:
            continue
        m = evaluate(pred[sel], targets[orden][sel])
        print(f"  {control:<12}{sel.sum():>12,}{m.rmse:>10.4f}{m.mae_cp:>9.1f}"
              f"{m.sign_agreement:>9.1%}")

control         posiciones      RMSE   MAE cp    signo
------------------------------------------------------
ResNet
  Blitz            116,471    0.2506    105.5    87.9%
  Rapid             10,603    0.2576    109.6    87.2%
  Classical            425    0.2255     86.1    84.4%
Transformer
  Blitz            116,471    0.2529    108.9    86.8%
  Rapid             10,603    0.2560    111.5    86.7%
  Classical            425    0.2288     85.1    84.4%


### 5.3 Acuerdo de signo según qué tan pareja está la posición

Esto no está en el WBS y lo agrego porque **predice cómo va a jugar el motor**.

La búsqueda elige entre jugadas cuyas posiciones resultantes suelen estar muy
cerca en valor. Un acuerdo de signo global del 87 % puede ser 95 % en posiciones
decididas y mucho menos en las igualadas — y el motor elige justo ahí. Si esa
segunda cifra es baja, la fuerza de juego va a ser peor de lo que sugiere
cualquier RMSE.

In [8]:
from chessdl.training.metrics import sign_agreement

cortes = [(0.00, 0.05, 'casi igualada'), (0.05, 0.20, 'ligera ventaja'),
          (0.20, 0.50, 'ventaja clara'), (0.50, 1.01, 'decidida')]

print(f"{'franja':<18}{'|valor|':>12}{'posiciones':>12}", end='')
for n in evaluadores: print(f"{n:>14}", end='')
print()
print('-' * (42 + 14 * len(evaluadores)))
for bajo, alto, etiqueta in cortes:
    reales = targets[orden]
    sel = (np.abs(reales) >= bajo) & (np.abs(reales) < alto)
    print(f"{etiqueta:<18}{f'{bajo:.2f}-{alto:.2f}':>12}{sel.sum():>12,}", end='')
    for nombre, pred in predicciones.items():
        # deadband=0: `sign_agreement` excluye por defecto las posiciones con
        # |valor| <= 0,05, que es *exactamente* la primera franja -- la mas
        # interesante de todas. Con el valor por defecto esa fila vuelve `nan`.
        print(f"{sign_agreement(pred[sel], reales[sel], deadband=0.0):>14.1%}", end='')
    print()

franja                 |valor|  posiciones        ResNet   Transformer
----------------------------------------------------------------------
casi igualada        0.00-0.05      19,340         60.2%         61.3%
ligera ventaja       0.05-0.20      33,256         77.6%         75.8%
ventaja clara        0.20-0.50      29,833         88.1%         87.4%
decidida             0.50-1.01      45,070         95.1%         94.5%


## 6. Requerimiento 1.7: tiempo por jugada según la profundidad

La búsqueda de un nivel es un forward pass sobre las jugadas legales. Cada ply
extra multiplica las posiciones por la ramificación, así que esta tabla es la que
decide a qué profundidades se pueden correr los torneos.

In [9]:
import chess
from chessdl.engine.search import search

PRESUPUESTO = 5.0   # segundos, requerimiento 1.7
prueba = [fens[i] for i in orden[:24]]

print(f"{'arquitectura':<14}{'prof':>6}{'hojas':>10}{'mediana ms':>12}{'p90 ms':>10}{'margen':>9}")
print('-' * 61)
factibles, costo = {}, {}
for nombre, ev in evaluadores.items():
    factibles[nombre] = []
    for profundidad in (1, 2, 3):
        tiempos, hojas = [], []
        for fen in prueba[:8 if profundidad == 3 else 24]:
            board = chess.Board(fen)
            if board.is_game_over():
                continue
            r = search(board, ev, depth=profundidad)
            tiempos.append(r.seconds); hojas.append(r.leaves)
        t = np.array(tiempos) * 1000
        p90 = float(np.percentile(t, 90))
        entra = p90 / 1000 < PRESUPUESTO
        if entra:
            factibles[nombre].append(profundidad)
        costo[(nombre, profundidad)] = float(np.median(t)) / 1000
        print(f"{nombre if profundidad == 1 else '':<14}{profundidad:>6}{int(np.mean(hojas)):>10,}"
              f"{np.median(t):>12.0f}{p90:>10.0f}"
              f"{(PRESUPUESTO*1000/p90):>8.0f}x" + ('' if entra else '   NO ENTRA'))

print()
for nombre, ds in factibles.items():
    print(f'{nombre}: profundidades dentro del presupuesto -> {ds or [1]}')
    factibles[nombre] = ds or [1]

arquitectura    prof     hojas  mediana ms    p90 ms   margen
-------------------------------------------------------------
ResNet             1        29           7        10     483x
                   2     1,015         157       230      22x
                   3    32,688        3583     10709       0x   NO ENTRA
Transformer        1        29           9        12     420x
                   2     1,015         321       443      11x
                   3    32,688        8184     19069       0x   NO ENTRA

ResNet: profundidades dentro del presupuesto -> [1, 2]
Transformer: profundidades dentro del presupuesto -> [1, 2]


## 7. Calidad de jugada: pérdida media en centipeones

La métrica principal del bloque. Para cada posición se le pregunta a Stockfish
cuánto vale su mejor jugada y cuánto vale la que eligió el motor; la diferencia
es lo que esa elección tiró.

Es la medición que permite comparar dos motores parecidos sin jugar miles de
partidas, y la que va a decir si el empate del bloque 4 —0,80 % de RMSE— se
traduce en algo sobre un tablero.

In [10]:
import chess.engine
from chessdl.engine.match import move_quality

POSICIONES_ACPL = 400        # subir si sobra tiempo; el intervalo se angosta con la raiz
PROF_REFERENCIA = 12         # la misma con la que se etiqueto el dataset

rng = np.random.default_rng(0)
muestra_acpl = [fens[i] for i in rng.choice(orden, size=POSICIONES_ACPL, replace=False)]
limite_sf = chess.engine.Limit(depth=PROF_REFERENCIA)

calidades = {}
with chess.engine.SimpleEngine.popen_uci('./bin/stockfish') as sf:
    for nombre, ev in evaluadores.items():
        for profundidad in factibles[nombre]:
            clave = f'{nombre} d{profundidad}'
            calidades[clave] = move_quality(
                ev, sf, muestra_acpl, limite_sf,
                depth=profundidad, label=clave, progress=True,
            )
            print(calidades[clave].summary()); print()

ResNet d1:   0%|          | 0/400 [00:00<?, ?pos/s]

ResNet d1: 400 posiciones
  perdida media       176.8 cp +- 14.4
  mediana             52.0 cp
  acuerdo con SF      30.5%
  errores > 300 cp    21.0%



ResNet d2:   0%|          | 0/400 [00:00<?, ?pos/s]

ResNet d2: 400 posiciones
  perdida media       87.4 cp +- 11.2
  mediana             25.0 cp
  acuerdo con SF      39.0%
  errores > 300 cp    5.5%



Transformer d1:   0%|          | 0/400 [00:00<?, ?pos/s]

Transformer d1: 400 posiciones
  perdida media       257.1 cp +- 18.6
  mediana             84.0 cp
  acuerdo con SF      28.5%
  errores > 300 cp    34.2%



Transformer d2:   0%|          | 0/400 [00:00<?, ?pos/s]

Transformer d2: 400 posiciones
  perdida media       97.9 cp +- 10.7
  mediana             25.0 cp
  acuerdo con SF      37.8%
  errores > 300 cp    8.0%



In [11]:
print(f"{'motor':<20}{'ACPL':>10}{'mediana':>10}{'acuerdo':>10}{'>300cp':>9}")
print('-' * 59)
for clave, q in sorted(calidades.items(), key=lambda kv: kv[1].acpl):
    print(f"{clave:<20}{q.acpl:>10.1f}{np.median(q.losses_cp):>10.1f}"
          f"{q.agreement:>10.1%}{q.blunder_rate():>9.1%}")

motor                     ACPL   mediana   acuerdo   >300cp
-----------------------------------------------------------
ResNet d2                 87.4      25.0     39.0%     5.5%
Transformer d2            97.9      25.0     37.8%     8.0%
ResNet d1                176.8      52.0     30.5%    21.0%
Transformer d1           257.1      84.0     28.5%    34.2%


## 8. Torneo A: Elo estimado

Contra Stockfish con la fuerza acotada por `UCI_Elo`, en escalones, para acotar
dónde cae el motor. Cada apertura se juega **dos veces con los colores
cambiados**: si el conjunto de aperturas favoreciera a las blancas, el sesgo se
cancela en vez de terminar entero en una columna.

**El torneo corre a profundidad 1**, que es la del requerimiento 1.6. No es una
limitación de tiempo por jugada —la tabla de la sección 6 muestra que sobra
margen— sino de tiempo total: una partida son unas 40 jugadas del motor, y a
profundidad 3 cada una cuesta segundos en vez de milisegundos. La celda que sigue
estima el costo antes de lanzar nada, y la sección 8.2 mide aparte si la
profundidad compra Elo, con muchas menos partidas.

In [12]:
from chessdl.engine.match import (
    estimated_seconds, opening_positions, play_match, stockfish_at_elo,
)

APERTURAS = 15               # x2 partidas cada una
ESCALONES = [1320, 1500, 1700]

# La profundidad del motor en la escalera. La primera corrida uso 1 -- la del
# requerimiento 1.6 -- y el resultado quedo subestimado: la seccion 7 mostro
# que la profundidad 2 parte al medio la perdida en centipeones, y la 6 que
# entra con 18x de margen. La celda de abajo estima el costo antes de lanzar.
PROF_ESCALERA = 2

aperturas = opening_positions([fens[i] for i in orden], plies[orden],
                              count=APERTURAS, max_ply=16, seed=0)
print(f'{len(aperturas)} aperturas del split de test, {2*len(aperturas)} partidas por escalon\n')

print('costo estimado del torneo completo, por profundidad:')
for profundidad in (1, 2, 3):
    total = sum(estimated_seconds(APERTURAS, costo[(n, profundidad)]) * len(ESCALONES)
                for n in evaluadores)
    print(f'  profundidad {profundidad}: {total/60:6.1f} min'
          + ('  <- la que se corre' if profundidad == PROF_ESCALERA else ''))

15 aperturas del split de test, 30 partidas por escalon

costo estimado del torneo completo, por profundidad:
  profundidad 1:    1.0 min
  profundidad 2:   28.7 min  <- la que se corre
  profundidad 3:  706.0 min


In [13]:
torneos = {}
with chess.engine.SimpleEngine.popen_uci('./bin/stockfish') as sf:
    for nombre, ev in evaluadores.items():
        for elo in ESCALONES:
            stockfish_at_elo(sf, elo)
            clave = f'{nombre} vs SF {elo}'
            torneos[clave] = play_match(
                # Limite de TIEMPO y no de profundidad: `UCI_Elo` esta calibrado
                # para busquedas con control de tiempo, y fijarle la profundidad
                # a mano pisa el mecanismo con el que Stockfish se debilita. Con
                # `depth=6` el rival juega debilitado, pero su Elo no es el numero
                # de la etiqueta -- que para la memoria es peor que no tenerlo.
                ev, sf, aperturas, chess.engine.Limit(time=0.05),
                depth=PROF_ESCALERA, label=clave, progress=True, max_plies=160,
            )
            print(torneos[clave].summary()); print()

ResNet vs SF 1320:   0%|          | 0/30 [00:00<?, ?partida/s]

ResNet vs SF 1320: 30 partidas
  22W 1T 7D
  puntos por partida  0.750 +- 0.079
  diferencia de Elo   +191
  largo medio         65 plies



ResNet vs SF 1500:   0%|          | 0/30 [00:00<?, ?partida/s]

ResNet vs SF 1500: 30 partidas
  8W 7T 15D
  puntos por partida  0.383 +- 0.078
  diferencia de Elo   -83
  largo medio         93 plies



ResNet vs SF 1700:   0%|          | 0/30 [00:00<?, ?partida/s]

...

Transformer vs SF 1320: 30 partidas
  12W 5T 13D
  puntos por partida  0.483 +- 0.085
  diferencia de Elo   -12
  largo medio         81 plies



Transformer vs SF 1500:   0%|          | 0/30 [00:00<?, ?partida/s]

Transformer vs SF 1500: 30 partidas
  4W 6T 20D
  puntos por partida  0.233 +- 0.067
  diferencia de Elo   -207
  largo medio         75 plies



Transformer vs SF 1700:   0%|          | 0/30 [00:00<?, ?partida/s]

Transformer vs SF 1700: 30 partidas
  9W 4T 17D
  puntos por partida  0.367 +- 0.083
  diferencia de Elo   -95
  largo medio         69 plies



### 8.2 ¿La profundidad compra Elo?

Menos partidas, contra un solo escalón, pero a cada profundidad que entró en el
presupuesto. Con este número de partidas el intervalo es ancho: sirve para ver un
salto grande, no para medir treinta puntos de Elo.

Lo interesante es **qué tan grande es el salto entre 1 y 2** comparado con el que
haya entre 2 y 3. La teoría dice que el primero debería ser el importante —es el
que cierra el punto ciego de la recaptura— y el segundo mucho menor por el mismo
precio multiplicado por treinta.

In [14]:
import math
ESCALON_PROF = 1500
APERTURAS_PROF = 6

aperturas_prof = opening_positions([fens[i] for i in orden], plies[orden],
                                   count=APERTURAS_PROF, max_ply=16, seed=1)
print('costo estimado:')
for nombre in evaluadores:
    for d in factibles[nombre]:
        print(f'  {nombre} d{d}: '
              f'{estimated_seconds(APERTURAS_PROF, costo[(nombre, d)])/60:.1f} min')
print()

por_profundidad = {}
with chess.engine.SimpleEngine.popen_uci('./bin/stockfish') as sf:
    stockfish_at_elo(sf, ESCALON_PROF)
    for nombre, ev in evaluadores.items():
        for profundidad in factibles[nombre]:
            clave = f'{nombre} d{profundidad}'
            por_profundidad[clave] = play_match(
                ev, sf, aperturas_prof, chess.engine.Limit(depth=6),
                depth=profundidad, label=clave, progress=True, max_plies=160,
            )

print(f"{'motor':<18}{'puntos':>10}{'+-':>8}{'W-T-D':>10}{'Elo implicito':>16}")
print('-' * 62)
for clave, t in por_profundidad.items():
    w, e, d = t.record
    elo = t.elo_difference()
    texto = 'sin acotar' if math.isinf(elo) else f'{ESCALON_PROF + elo:.0f}'
    print(f'{clave:<18}{t.score_rate:>10.3f}{t.margin():>8.3f}'
          f'{f"{w}-{e}-{d}":>10}{texto:>16}')
print()
print(f'Con {2*APERTURAS_PROF} partidas por brazo el Elo implicito es orientativo: la tasa de')
print('puntos trae +-0,1 y eso son mas de cien puntos de Elo. Para comparar las dos')
print('arquitecturas manda la perdida en centipeones de la seccion 7, que sale de')
print('400 posiciones y no de una docena de partidas.')

costo estimado:
  ResNet d1: 0.1 min
  ResNet d2: 1.3 min
  Transformer d1: 0.1 min
  Transformer d2: 2.6 min



ResNet d1:   0%|          | 0/12 [00:00<?, ?partida/s]

ResNet d2:   0%|          | 0/12 [00:00<?, ?partida/s]

Transformer d1:   0%|          | 0/12 [00:00<?, ?partida/s]

Transformer d2:   0%|          | 0/12 [00:00<?, ?partida/s]

motor                 puntos      +-     W-T-D   Elo implicito
--------------------------------------------------------------
ResNet d1              0.208   0.074     0-5-7            1268
ResNet d2              0.500   0.107     3-6-3            1500
Transformer d1         0.208   0.074     0-5-7            1268
Transformer d2         0.375   0.125     3-3-6            1411

Con 12 partidas por brazo el Elo implicito es orientativo: la tasa de
puntos trae +-0,1 y eso son mas de cien puntos de Elo. Para comparar las dos
arquitecturas manda la perdida en centipeones de la seccion 7, que sale de
400 posiciones y no de una docena de partidas.


## 9. Torneo B: contra Stockfish a un nivel

El torneo que responde la pregunta del trabajo. Los dos motores hacen **la misma
búsqueda** —un ply—, así que lo único que queda distinto es la función de
evaluación: una aprendida de 2,3 millones de posiciones, la otra escrita a mano
durante veinte años.

In [15]:
from chessdl.engine.match import stockfish_full_strength

torneos_1ply = {}
with chess.engine.SimpleEngine.popen_uci('./bin/stockfish') as sf:
    stockfish_full_strength(sf)
    for nombre, ev in evaluadores.items():
        clave = f'{nombre} d1 vs SF d1'
        torneos_1ply[clave] = play_match(
            ev, sf, aperturas, chess.engine.Limit(depth=1),
            depth=1, label=clave, progress=True,
        )
        print(torneos_1ply[clave].summary()); print()

ResNet d1 vs SF d1:   0%|          | 0/30 [00:00<?, ?partida/s]

ResNet d1 vs SF d1: 30 partidas
  0W 17T 13D
  puntos por partida  0.283 +- 0.046
  diferencia de Elo   -161
  largo medio         63 plies



Transformer d1 vs SF d1:   0%|          | 0/30 [00:00<?, ?partida/s]

Transformer d1 vs SF d1: 30 partidas
  0W 15T 15D
  puntos por partida  0.250 +- 0.046
  diferencia de Elo   -191
  largo medio         62 plies



## 10. Resumen

In [16]:
import math

print('REQUERIMIENTOS')
print('-' * 64)
print('1.6  seleccion por busqueda de un nivel          cumplido')
tiempos_reales = [g.seconds_per_move for t in torneos.values() for g in t.games
                  if not math.isnan(g.seconds_per_move)]
peor = np.percentile(tiempos_reales, 90) if tiempos_reales else float('nan')
print(f'1.7  tiempo por jugada (p90 en partida real)     {peor*1000:.0f} ms de 5000')
print('4.2  evaluacion en centipeones y vista blancas   cumplido (CLI y motor)')
print('4.7  desglose y tiempo de inferencia             secciones 5 y 6')
print()

print('QUE COMPRA LA PROFUNDIDAD')
print('-' * 64)
print(f"{'':<16}{'ACPL d1':>10}{'ACPL d2':>10}{'errores >300cp':>17}")
for nombre in evaluadores:
    q1, q2 = calidades[f'{nombre} d1'], calidades[f'{nombre} d2']
    print(f'{nombre:<16}{q1.acpl:>10.1f}{q2.acpl:>10.1f}'
          f'{f"{q1.blunder_rate():.0%} -> {q2.blunder_rate():.0%}":>17}')
print()
print('Un ply mas parte al medio la perdida y reduce los errores graves a un')
print('cuarto. Es el salto que cierra el punto ciego de la recaptura, y cuesta')
print('285 ms por jugada: 18 veces por debajo del presupuesto del 1.7.')
print()

print('LOS DOS MOTORES')
print('-' * 64)
print(f"{'':<16}{'prof':>6}{'ACPL':>9}{'acuerdo':>10}{'vs SF 1320':>13}{'Elo aprox':>12}")
for nombre in evaluadores:
    t = torneos[f'{nombre} vs SF 1320']
    elo = t.elo_difference()
    texto = 'sin acotar' if math.isinf(elo) else f'{1320 + elo:.0f}'
    for profundidad in factibles[nombre]:
        q = calidades[f'{nombre} d{profundidad}']
        marca = f'{t.score_rate:>13.3f}{texto:>12}' if profundidad == PROF_ESCALERA else ''
        print(f'{nombre if profundidad == 1 else "":<16}{profundidad:>6}'
              f'{q.acpl:>9.1f}{q.agreement:>10.1%}{marca}')
print()

print('LA DIFERENCIA ENTRE ARQUITECTURAS, SEGUN LA PROFUNDIDAD')
print('-' * 64)
for profundidad in sorted(set(factibles['ResNet']) & set(factibles['Transformer'])):
    a = calidades[f'ResNet d{profundidad}']
    b = calidades[f'Transformer d{profundidad}']
    d = a.acpl - b.acpl
    error = np.hypot(*[c.losses_cp.std(ddof=1) / np.sqrt(len(c.losses_cp))
                       for c in (a, b)])
    separadas = abs(d) >= 2 * error
    print(f'  profundidad {profundidad}: {abs(d):>5.1f} cp +-{error:>4.1f} a favor de '
          f'{"Transformer" if d > 0 else "ResNet":<12}'
          f'{"SE DISTINGUEN" if separadas else "dentro del error"}')
print()
print('Esa reversion es el resultado mas interesante del bloque. Con un ply, la')
print('ResNet le saca ventaja clara: la busqueda no corrige nada y manda la')
print('evaluacion cruda. Con dos, la brecha se vuelve indistinguible -- la')
print('busqueda rescata los errores locales de la red mas ruidosa.')
print()
print('O sea que "cual arquitectura es mejor" no tiene una respuesta sola: depende')
print('de cuanta busqueda haya encima. El empate por RMSE del bloque 4 predijo bien')
print('el caso con busqueda, y mal el caso sin ella.')
print()
print(f'Pesos y metricas: https://huggingface.co/{repo_modelos}')

REQUERIMIENTOS
----------------------------------------------------------------
1.6  seleccion por busqueda de un nivel          cumplido
1.7  tiempo por jugada (p90 en partida real)     358 ms de 5000
4.2  evaluacion en centipeones y vista blancas   cumplido (CLI y motor)
4.7  desglose y tiempo de inferencia             secciones 5 y 6

QUE COMPRA LA PROFUNDIDAD
----------------------------------------------------------------
                   ACPL d1   ACPL d2   errores >300cp
ResNet               176.8      87.4        21% -> 6%
Transformer          257.1      97.9        34% -> 8%

Un ply mas parte al medio la perdida y reduce los errores graves a un
cuarto. Es el salto que cierra el punto ciego de la recaptura, y cuesta
285 ms por jugada: 18 veces por debajo del presupuesto del 1.7.

LOS DOS MOTORES
----------------------------------------------------------------
                  prof     ACPL   acuerdo   vs SF 1320   Elo aprox
ResNet               1    176.8     30.5%
         

**Próximo paso:** ninguno por ahora. El bloque 6 cierra el alcance de este
trabajo; la continuación —más posiciones etiquetadas, reentrenar las dos
arquitecturas sobre ese dataset y compararlas contra aprendizaje por refuerzo—
queda anotada como trabajo futuro en [`docs/modelado.md`](../docs/modelado.md).